# <div style="text-align:center; border-radius:15px; padding:15px; margin:0; font-size:100%; font-family: 'Arial Black', Arial, sans-serif; text-shadow: 2px 2px 5px rgba(0,0,0,0.7); background: linear-gradient(90deg, #0f2027, #203a43, #2c5364); overflow:hidden; box-shadow:0 2px 5px rgba(0, 0, 0, 0.3); color:white;"><b>3.1 Load Data</b></div>

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from joblib import dump
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna.visualization as vis

In [19]:
X_train, X_test, y_train, y_test = joblib.load("../models/train_test_split.pkl")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (72836, 28)
X_test shape: (18210, 28)
y_train shape: (72836,)
y_test shape: (18210,)


# <div style="text-align:center; border-radius:15px; padding:15px; margin:0; font-size:100%; font-family: 'Arial Black', Arial, sans-serif; text-shadow: 2px 2px 5px rgba(0,0,0,0.7); background: linear-gradient(90deg, #0f2027, #203a43, #2c5364); overflow:hidden; box-shadow:0 2px 5px rgba(0, 0, 0, 0.3); color:white;"><b>3.2 Hyperparameter Tuning with Optuna</b></div>

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def xgb_objective(trial):
    params = {
        # Number of boosting rounds
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 1000),
        # Maximum depth of each tree
        'max_depth'       : trial.suggest_int('max_depth', 3, 15),
        # Step size shrinkage
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        # Fraction of training samples per tree
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        # Fraction of features per tree
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        # Minimum instance weight in a leaf
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        # L2 regularization
        'reg_lambda'      : trial.suggest_float('reg_lambda', 0.0, 5.0),
        # L1 regularization
        'reg_alpha'       : trial.suggest_float('reg_alpha', 0.0, 5.0),

    }
    model = XGBClassifier(
        **params,
        scale_pos_weight=pos_weight,
        random_state=42,
        eval_metric='logloss'
    )
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='roc_auc', n_jobs=-1)
    return np.mean(scores)

sampler = optuna.samplers.TPESampler(seed=42)
study   = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(xgb_objective, n_trials=150)

print(f'Best CV AUC    : {study.best_value:.4f}')
print(f'Best XGB Params: {study.best_params}')


Best CV AUC    : 0.9557
Best XGB Params: {'n_estimators': 943, 'max_depth': 5, 'learning_rate': 0.046590058103669396, 'subsample': 0.9648549173887546, 'colsample_bytree': 0.9957609029778385, 'min_child_weight': 9, 'reg_lambda': 1.1126684305635504, 'reg_alpha': 0.16981486696331743}


In [21]:
vis.plot_optimization_history(study).show()


vis.plot_param_importances(study).show()

In [22]:
pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_verify = XGBClassifier(
    **study.best_params,
    scale_pos_weight=pos_weight,
    random_state=42,
    eval_metric='logloss'
)

verify_scores = cross_val_score(
    xgb_verify, X_train, y_train,
    cv=kf, scoring='roc_auc', n_jobs=-1
)

print(f'Tuned XGBoost CV AUC Scores : {verify_scores}')
print(f'Mean AUC : {np.mean(verify_scores):.4f}')
print(f'Std  AUC : {np.std(verify_scores):.4f}')


Tuned XGBoost CV AUC Scores : [0.95589766 0.95674913 0.95534943 0.95666143 0.9537438 ]
Mean AUC : 0.9557
Std  AUC : 0.0011


In [23]:
joblib.dump(study.best_params, "../models/xgb_best_params.pkl")
print("Best parameters saved to ../models/xgb_best_params.pkl")

Best parameters saved to ../models/xgb_best_params.pkl
